# Ejercicio 4. Descenso de gradiente con momento
## 1. Función y gradiente
La función de Rosenbrock es $f(x_1,x_2)=(1-x_1)^2+100(x_2-x_1^2)^2$.

Sus derivadas parciales son:
$$\frac{\partial f}{\partial x_1}=2(x_1-1)-400x_1(x_2-x_1^2),\qquad \frac{\partial f}{\partial x_2}=200(x_2-x_1^2).$$

Se utiliza momento clásico: $v_{i+1}=\alpha v_i-\gamma\nabla f(x_i)$, $x_{i+1}=x_i+v_{i+1}$, con $v_0=0$. Para $\alpha=0$ se recupera el método convencional. El mínimo global exacto es $(1,1)$, con valor cero, porque ambos sumandos son cuadrados no negativos.

In [1]:
import numpy as np

def f(x):
 a,b=x; return (1-a)**2+100*(b-a*a)**2

def gradiente(x):
 a,b=x; return np.array([2*(a-1)-400*a*(b-a*a),200*(b-a*a)])

def descenso(gamma,alpha,x0,tol=1e-6,max_iter=100000):
 x=np.array(x0,dtype=float); v=np.zeros_like(x); trayectoria=[x.copy()]; valores=[f(x)]
 for i in range(max_iter+1):
  g=gradiente(x)
  if np.linalg.norm(g)<tol: estado='Converge'; break
  if i==max_iter: estado='Límite de iteraciones'; break
  v=alpha*v-gamma*g; x=x+v
  if not np.all(np.isfinite(x)) or np.linalg.norm(x)>1e6:
   estado='Diverge'; break
  trayectoria.append(x.copy()); valores.append(f(x))
 return dict(x=trayectoria[-1],g=gradiente(trayectoria[-1]),f=valores[-1],n=len(valores)-1,estado=estado,trayectoria=np.array(trayectoria),valores=np.array(valores))


import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


## 2. Selección heurística
Se prueban distintas tasas, momentos y puntos iniciales. Se limita cada ejecución a 100 000 actualizaciones, suficiente para comparar con el método convencional y evitar ciclos indefinidos. Se interrumpe una trayectoria si su norma supera $10^6$ o aparecen valores no finitos; en ese caso se reporta el último punto aceptado. Los incrementos de la función permiten detectar comportamiento no monótono, aunque no prueban por sí solos una oscilación.

In [2]:
pruebas = [(.0001,0,(-1.2,1)),(.001,0,(-1.2,1)),(.001,.5,(-1.2,1)),(.001,.9,(-1.2,1)),(.002,.9,(-1.2,1)),(.005,.9,(-1.2,1)),(.01,.9,(-1.2,1)),(.001,.99,(-1.2,1)),(.001,.9,(0,0)),(.001,.9,(2,2))]
filas=[]
for gamma, alpha, x0 in pruebas:
    r=descenso(gamma,alpha,x0)
    filas.append([gamma,alpha,x0,r['n'],r['estado'],r['f'],np.linalg.norm(r['g']),np.sum(np.diff(r['valores'])>1e-12)])
display(pd.DataFrame(filas,columns=['γ','α','x0','Actualizaciones aceptadas','Estado','f final','Norma gradiente','Incrementos de f']))

,γ,α,x0,Actualizaciones aceptadas,Estado,f final,Norma gradiente,Incrementos de f
0,0.0001,0.00,"(-1.2, 1)",100000,Límite de iteraciones,5.887020e-05,6.899410e-03,0
1,0.0010,0.00,"(-1.2, 1)",32076,Converge,1.251724e-12,9.998902e-07,0
2,0.0010,0.50,"(-1.2, 1)",16014,Converge,1.250327e-12,9.993322e-07,4
3,0.0010,0.90,"(-1.2, 1)",3020,Converge,1.248244e-12,9.984995e-07,17
4,0.0020,0.90,"(-1.2, 1)",1549,Converge,1.245895e-12,9.975576e-07,42
5,0.0050,0.90,"(-1.2, 1)",6,Diverge,8.771939e+25,3.625615e+20,5
6,0.0100,0.90,"(-1.2, 1)",3,Diverge,1.542582e+14,5.536830e+11,3
7,0.0010,0.99,"(-1.2, 1)",3185,Converge,2.638155e-15,8.844722e-07,1450
8,0.0010,0.90,"(0, 0)",2996,Converge,1.246768e-12,9.979090e-07,0
9,0.0010,0.90,"(2, 2)",3048,Converge,1.251228e-12,9.996922e-07,49


Se selecciona γ=0.001, α=0.9 y x0=(-1.2,1). Este punto está fuera del valle y permite evaluar la aceleración en una situación exigente. Aunque γ=0.002 converge en menos iteraciones con momento, γ=0.001 presenta menos incrementos de la función y permite una comparación estable con el método convencional. α=0.99 produce muchos más incrementos y tarda más. Las tasas 0.005 y 0.01 divergen en las pruebas realizadas; 0.0001 sin momento converge demasiado lentamente para el límite establecido.

## 3. Resultados y comparación con el método sin momento

In [3]:
gamma=0.001
x0=(-1.2,1)
con=descenso(gamma,0.9,x0)
sin=descenso(gamma,0,x0)
np.set_printoptions(precision=12)
for nombre,r in [('Con momento',con),('Sin momento',sin)]:
    print(nombre, '\n x* =',r['x'],'\n f(x*) =',r['f'],'\n Gradiente =',r['g'],'\n Norma =',np.linalg.norm(r['g']),'\n Iteraciones =',r['n'],'\n Estado =',r['estado'],'\n')
    assert np.linalg.norm(r['g'])<1e-6
print('Factor de reducción de iteraciones:',sin['n']/con['n'])
print('Reducción porcentual:',100*(1-con['n']/sin['n']))

Con momento 
 x* = [0.999998883645 0.999997762824] 
 f(x*) = 1.2482441843075059e-12 
 Gradiente = [-4.458291353341e-07 -8.934414763573e-07] 
 Norma = 9.984995190726152e-07 
 Iteraciones = 3020 
 Estado = Converge 

Sin momento 
 x* = [0.99999888209  0.999997759708] 
 f(x*) = 1.2517236788075983e-12 
 Gradiente = [-4.464501961018e-07 -8.946857921188e-07] 
 Norma = 9.998902160830393e-07 
 Iteraciones = 32076 
 Estado = Converge 

Factor de reducción de iteraciones: 10.621192052980133
Reducción porcentual: 90.58486095523133


## 4. Gráfica de la función contra la iteración

In [4]:
plt.figure(figsize=(10,5))
for nombre,r in [('Con momento',con),('Sin momento',sin)]:
    plt.semilogy(np.arange(len(r['valores'])),r['valores'],label=nombre)
plt.xlabel('Iteración i'); plt.ylabel('f(x_i) (escala logarítmica)'); plt.title('Comparación de convergencia'); plt.legend(); plt.grid(alpha=.3); plt.show()

/tmp/ipykernel_933/871410546.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel('Iteración i'); plt.ylabel('f(x_i) (escala logarítmica)'); plt.title('Comparación de convergencia'); plt.legend(); plt.grid(alpha=.3); plt.show()


## 5. Comparación de trayectorias

In [5]:
X,Y=np.meshgrid(np.linspace(-1.5,1.3,400),np.linspace(-.5,1.8,400))
Z=(1-X)**2+100*(Y-X**2)**2
plt.figure(figsize=(9,6))
plt.contour(X,Y,Z,levels=np.logspace(-2,3,22),alpha=.55)
for nombre,r in [('Sin momento',sin),('Con momento',con)]:
    t=r['trayectoria']; plt.plot(t[:,0],t[:,1],label=nombre,linewidth=1.5)
plt.scatter(*x0,marker='o',label='Inicio'); plt.scatter(1,1,marker='*',s=120,label='Mínimo exacto'); plt.xlabel('x1'); plt.ylabel('x2'); plt.title('Trayectorias sobre curvas de nivel'); plt.legend(); plt.grid(alpha=.2); plt.show()

/tmp/ipykernel_933/1789381895.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.scatter(*x0,marker='o',label='Inicio'); plt.scatter(1,1,marker='*',s=120,label='Mínimo exacto'); plt.xlabel('x1'); plt.ylabel('x2'); plt.title('Trayectorias sobre curvas de nivel'); plt.legend(); plt.grid(alpha=.2); plt.show()


## Conclusión
Con los parámetros seleccionados, el momento reduce las iteraciones de 32 076 a 3 020 (90.59 % menos, aproximadamente 10.62 veces). Ambos métodos satisfacen la misma tolerancia y alcanzan prácticamente el mismo mínimo. El momento acelera el avance a lo largo del valle, aunque puede causar sobrepasos y aumentos transitorios de la función. Esta ventaja depende de los parámetros y no garantiza una mejora para cualquier problema.